# 08 - Hyperparameter Tuning, Done Correctly

Notebook 01 called `RandomizedSearchCV` and reported the best score it found.
**That reported score is optimistically biased** - you searched hundreds of settings and
kept the luckiest. This notebook shows the correct procedure.

By the end you will be able to:
1. Distinguish parameters from hyperparameters.
2. Choose between grid search and randomised search.
3. Tune *inside* a pipeline, so preprocessing never leaks.
4. Keep an honest final estimate of performance.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
np.random.seed(42)

heart = pd.read_csv("../data/heart-disease.csv")
X = heart.drop("target", axis=1)
y = heart["target"]
print(X.shape, y.value_counts().to_dict())

## 1. Parameters vs hyperparameters

- **Parameters** are learned by `.fit()` - the coefficients of a regression, the splits
  in a tree. You never set these.
- **Hyperparameters** are set by you *before* fitting - `max_depth`, `alpha`,
  `n_neighbors`, `learning_rate`. These are what you tune.

## 2. Set aside a final test set first

This split happens once, at the start, and the test set is not touched again until the
very last cell. Every tuning decision is made using cross-validation on the training set.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

cv = StratifiedKFold(5, shuffle=True, random_state=42)
print(f"training set: {len(X_train)} rows   |   held-out test set: {len(X_test)} rows")
print("The test set is now locked away until the final cell.")

## 3. Grid search - exhaustive

`GridSearchCV` tries **every combination**. Reliable, but the cost explodes:
3 x 4 x 3 values with 5-fold CV is 36 x 5 = 180 model fits.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

grid = {
    "n_estimators": [100, 300],
    "max_depth": [3, 5, 10, None],
    "min_samples_leaf": [1, 2, 4],
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=grid, cv=cv, scoring="roc_auc",
    n_jobs=-1, return_train_score=True)

gs.fit(X_train, y_train)

print(f"combinations tried : {len(gs.cv_results_['params'])}")
print(f"best CV roc_auc    : {gs.best_score_:.3f}")
print(f"best parameters    : {gs.best_params_}")

In [ ]:
# Always look at the top few, not just the winner.
results = pd.DataFrame(gs.cv_results_)
cols = ["param_n_estimators", "param_max_depth", "param_min_samples_leaf",
        "mean_test_score", "std_test_score"]
results[cols].sort_values("mean_test_score", ascending=False).head(8).round(4)

> **Look at `std_test_score`.** If the top five settings are within one standard deviation
> of each other, they are tied. Pick the **simplest** one - a shallower tree, fewer
> estimators - not the one that happened to score 0.001 higher.

## 4. Randomised search - usually better value

`RandomizedSearchCV` samples `n_iter` random combinations from distributions. For the same
compute budget it usually finds a better setting, because most hyperparameters barely
matter and grid search wastes fits exploring them exhaustively.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

distributions = {
    "n_estimators": randint(100, 500),
    "max_depth": [3, 5, 10, 15, None],
    "min_samples_leaf": randint(1, 10),
    "max_features": uniform(0.1, 0.9),
}

rs = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=distributions, n_iter=40,
    cv=cv, scoring="roc_auc", n_jobs=-1, random_state=42)

rs.fit(X_train, y_train)
print(f"fits used       : {40 * 5}  (grid search used {len(gs.cv_results_['params']) * 5})")
print(f"best CV roc_auc : {rs.best_score_:.3f}")
print(f"best parameters : {rs.best_params_}")

**Practical workflow:** randomised search wide first to find the promising region, then a
small grid search around the winner to refine. Do not start with a huge grid.

## 5. Tuning inside a pipeline

This is the part that matters most. When preprocessing is in the pipeline, it is refit on
each CV fold - so tuning cannot leak (notebook 04). Address a step's hyperparameter with
`stepname__paramname`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("select", SelectKBest(f_classif)),
    ("clf", LogisticRegression(max_iter=5000)),
])

pipe_grid = {
    "select__k": [5, 8, 10, 13],
    "clf__C": [0.01, 0.1, 1, 10, 100],
}

pipe_search = GridSearchCV(pipe, pipe_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
pipe_search.fit(X_train, y_train)

print(f"best CV roc_auc : {pipe_search.best_score_:.3f}")
print(f"best parameters : {pipe_search.best_params_}")

Note `select__k` and `clf__C`: the double underscore reaches into a named step. You can
even search over the *model itself* by putting a list of estimators in the grid.

## 6. The honest final number

`best_score_` is biased upward - it is the maximum over many noisy estimates. The
unbiased number comes from the test set you locked away in section 2, used exactly once.

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

best_model = pipe_search.best_estimator_          # already refit on all of X_train
test_auc = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])

print(f"CV score during tuning (optimistic) : {pipe_search.best_score_:.3f}")
print(f"held-out test score (honest)        : {test_auc:.3f}")
print()
print(classification_report(y_test, best_model.predict(X_test),
                            target_names=["no disease", "disease"], digits=3))

The honest number is usually lower. **That drop is normal** - report the held-out number,
not the tuning number.

### When you cannot spare a test set: nested CV

With very little data, a held-out set is too small to trust. **Nested cross-validation**
puts the whole search inside an outer CV loop. It is expensive but gives an unbiased
estimate of "how well does my *tuning procedure* generalise".

In [ ]:
from sklearn.model_selection import cross_val_score

inner = StratifiedKFold(3, shuffle=True, random_state=1)
outer = StratifiedKFold(5, shuffle=True, random_state=2)

search = GridSearchCV(pipe, pipe_grid, cv=inner, scoring="roc_auc", n_jobs=-1)
nested = cross_val_score(search, X, y, cv=outer, scoring="roc_auc")

print(f"nested CV roc_auc: {nested.mean():.3f} +/- {nested.std():.3f}")
print("This is an unbiased estimate of the whole tune-and-fit procedure.")

## 7. Saving the finished model

In [ ]:
import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)
joblib.dump(best_model, "models/heart_disease_pipeline.joblib")

# The saved object contains the scaler, the selector AND the classifier -
# so it takes raw dataframes at predict time. That is the point of pipelines.
reloaded = joblib.load("models/heart_disease_pipeline.joblib")
print("reloaded test AUC:", round(roc_auc_score(y_test, reloaded.predict_proba(X_test)[:, 1]), 3))
print("\nsample prediction on raw data:", reloaded.predict(X_test.head(3)))

---
## Exercises

1. Time both searches with `%%time`. How much slower is grid search, and did it find a
   better setting?
2. Add `"clf__class_weight": [None, "balanced"]` to `pipe_grid`. Does the tuner choose it?
   Now re-tune with `scoring="average_precision"` and see whether the answer changes.
3. Look at the top 10 rows of `pipe_search.cv_results_`. Are the scores meaningfully
   different, or all within one std? Which setting would you actually ship?
4. Run the same tuning with `scoring="recall"` instead of `"roc_auc"`. How do the chosen
   hyperparameters change, and what does that tell you about the choice of metric?
5. **Stretch:** search over two different classifiers in one go by putting
   `{"clf": [LogisticRegression(max_iter=5000), RandomForestClassifier()]}` in the grid
   as a list of dicts, each with its own parameters.